# 05 The panel: the target, the transform, and the checks of coverage and look-ahead

**What this notebook does.** It joins the twelve characteristics of notebooks 03 and 04 with the target, the return each member earned in month t above one-month Treasury bills, into the panel the models will read. It judges coverage, gate G1 (b). It ranks each characteristic within its month onto the interval from -1 to 1 and sets a missing value to the month's median. And it builds 24 months again from inputs cut at the last trading day of month t-1, to show that no characteristic used anything later, gate G1 (c).

**Why it comes here.** A characteristic missing for a large share of members would leave the models little to learn from, and one that used a later number would make every forecast look better than any investor could have made it, with nothing in the results to show it.

**What it judges.** Gate G1 (b) and (c). With part (a) from notebook 02, all three must be met before milestone 2 fits any model.

**The data.** CRSP and Compustat, from Wharton Research Data Services, accessed through Queen Mary University of London, extracted on 28 September 2026, and Kenneth French's data library, downloaded at run time.

## Terms used in this notebook

| Term | Meaning |
|---|---|
| panel | one row per member and month t, January 1991 to December 2024, with the twelve characteristics measured at the end of month t-1 and the target of month t |
| risk-free rate | the return on one-month Treasury bills, which French's library gives month by month as the one-month bill rate |
| excess return | a return less the risk-free rate of the same month |
| target | the member's excess return in month t: its total return, with the delisting rule applied, less French's one-month bill rate of month t |
| one-month bill rate | French's monthly return on one-month Treasury bills, the risk-free rate of the month |
| coverage | the share of a month's members with a characteristic present, before any value is filled in |
| rank | a value's position when the month's values are sorted, from 1 for the smallest. Tied values share the average of the positions they occupy |
| rank transform | each characteristic ranked within its month and mapped to the interval from -1 to 1: rank r of n present values becomes -1 + 2 (r - 1) / (n - 1) |
| fill | a missing value set to 0 after the rank transform. On that scale 0 is the month's median |
| seed | the number that starts the random generator, 20261119 in the constants, so that every number it gives can be made again |
| look-ahead test | 24 of the panel's 408 months, drawn by the study's seed, built again from inputs from which every row dated after the last trading day of month t-1 is removed, and compared with the full build |
| tolerance | the largest difference allowed between the two builds, 1e-12 |
| verdict | met or not met: what a gate's level says of a result |
| gate G1 | (a) the copied layer gives Stock-Level Tracking's counts, judged in notebook 02. (b) Each characteristic is present for at least 90 per cent of each month's members, book-to-market and operating profitability for at least 80 per cent, in every month from January 1991 to December 2024. (c) The look-ahead test passes on its 24 months |

## What I expect to see, and where it comes from

1. 204,520 member-months, of which 100 have no target, and 114,521 in the test window: the counts of notebook 02 and the constants.
2. Gate G1 (b) is met: every characteristic at or above its level in all 408 months. The lowest coverage is expected in book-to-market and operating profitability, which need a positive book equity, and in the early 1990s, when Compustat holds fewer years of history for each company. Notebooks 03 and 04 give the yearly figures.
3. Gate G1 (c) is met with a largest difference of exactly 0. The second build runs the same three functions as the full build, on the same numbers up to the cut, so it gives the same results to the last bit. A difference above 1e-12, or a value present in one build and missing in the other, would mean a characteristic used a row from after the cut.
4. After the transform, each characteristic lies between -1 and 1 in every member-month, the count of filled cells equals the count of missing values before the fill, and each month's values average 0. Ranks with ties at their average always sum to n (n + 1) / 2, which the map turns into a mean of 0, and a filled cell adds a 0.

Running time: about five minutes, most of it spent reading the daily file again for the look-ahead test.

In [ ]:
import hashlib
import json
import pathlib
import sys
import time

# The package lives in src/ beside this folder. The notebook finds the repository's root from where it runs.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "mlrp").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from mlrp import constants as C
from mlrp import data as D
from mlrp.source import extraction as X
from mlrp.source import layer as L
from mlrp.source import returns as R
from mlrp.source import universe as U

# Stops with one sentence when the data root is not set, when the manifest is not the one the constants fix,
# or when one of the study's eight tables is not the one they record.
root, manifest = D.open_extraction()
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}", "| entries of the eight tables", C.MANIFEST_ENTRIES_SHA256[:12])

In [ ]:
g1a = D.read_result("g1a.json")
if not g1a["g1a_met"]:
    raise SystemExit("Gate G1 (a) is not met, so nothing after notebook 02 runs.")
print("Gate G1 (a): met, in notebook 02")

## 1. The target

The returns of notebook 02 are built again with the copied `monthly_panel`. French's monthly file gives the one-month bill rate, and `panel.target` subtracts it from the total return.

In [ ]:
from mlrp import characteristics as K
from mlrp import panel as P
universe = L.load_universe(root)
ulong = universe["ulong"]
monthly, _ = D.read_table(root, "crsp_msf")
monthly = monthly[monthly["permno"].isin(universe["matrix"].columns)].reset_index(drop=True)
delistings, _ = D.read_table(root, "crsp_msedelist")
returns_panel = R.monthly_panel(ulong, monthly, delistings)
_, bill_rate, french_monthly_records = D.french_monthly()
targets = P.target(returns_panel, bill_rate)
print("bill rate from", bill_rate.index.min(), "to", bill_rate.index.max())

## 2. The panel, before the transform

The characteristics kept by notebooks 03 and 04 are read from `local/` and joined with the target onto every member-month of the window.

In [ ]:
def from_local(name):
    frame = D.read_local(name)
    return frame.assign(month=pd.PeriodIndex(frame["month"], freq="M"))

market = from_local("market_characteristics.parquet")
accounting = from_local("accounting_characteristics.parquet")
monthly_part = market[["month", "permno", *K.MONTHLY]]
daily_part = market[["month", "permno", *K.DAILY]]
panel = P.assemble(ulong, monthly_part, daily_part, accounting, targets)
test = panel[pd.PeriodIndex(panel["month"], freq="M") >= pd.Period(C.TEST_FIRST_MONTH, "M")]
counts = {"member_months": int(len(panel)), "without_target": int(panel["target"].isna().sum()),
          "months": int(panel["month"].nunique()), "test_member_months": int(len(test))}
print(pd.DataFrame({"result": counts, "expected": {"member_months": C.PANEL_MEMBER_MONTHS, "without_target": C.PANEL_MEMBER_MONTHS_NO_RETURN,
                                                   "months": C.PANEL_MONTHS, "test_member_months": C.TEST_MEMBER_MONTHS}}).to_string())

## 3. Coverage and gate G1 (b)

The share of each month's members with each characteristic present, before the fill, in each of the 408 months. The gate asks every characteristic to be at or above its level in every month. The table gives each characteristic's lowest month.

In [ ]:
coverage = P.coverage_by_month(panel)
g1b = P.g1b(coverage)
print(pd.DataFrame(g1b["by_characteristic"]).T.to_string())
print("Gate G1 (b):", "met" if g1b["g1b_met"] else "not met", f"over {g1b['months']} months")
missing = P.missing_by_year(panel)

## 4. The rank transform and the fill

Within each month, each characteristic is ranked across the members for which it is present and mapped to the interval from -1 to 1. A missing value is then set to 0, the month's median on that scale.

**A worked example.** In one month five members have momentum of 0.30, -0.10, 0.05, 0.05 and a missing value. The four present values ranked from the smallest are -0.10 (rank 1), the two values of 0.05 (ranks 2 and 3, so 2.5 each) and 0.30 (rank 4). With n = 4, rank r becomes -1 + 2 (r - 1) / 3: -1, 0, 0 and 1. The missing value becomes 0. So the largest momentum of the month becomes 1 and the smallest -1, whatever their size, unless two members tie there, as two share classes of one company can on an accounting characteristic. A model sees only the order of the members within the month.

In [ ]:
transformed, filled = P.transform(panel)
names = list(C.CHARACTERISTICS)
checks = {
    "within -1 and 1": bool(transformed[names].apply(lambda s: s.between(C.RANK_LOW, C.RANK_HIGH)).all().all()),
    "filled cells equal the missing values": all(filled[c] == int(panel[c].isna().sum()) for c in names),
    # 1e-12: far above the rounding error of a mean of about 500 values between -1 and 1.
    "each month averages 0": bool((transformed.groupby("month")[names].mean().abs() <= 1e-12).all().all()),
}
print(json.dumps(checks, indent=1))
print("filled cells by characteristic:", filled)
print("largest absolute mean of a month:", float(transformed.groupby("month")[names].mean().abs().max().max()))

## 5. The look-ahead test, gate G1 (c)

For each of the 24 months drawn by the seed, the inputs are cut at the last trading day of month t-1. Every row dated later is removed from the monthly file, the delistings, the daily returns, French's daily series and the links, a link's end date after that day is set to missing, and every annual record less than six months old on that day is removed. The twelve characteristics of the month's members are then built again by the same three functions and compared with the full build, cell by cell.

In [ ]:
t0 = time.time()
months = P.lookahead_months()
days = ulong.drop_duplicates("month").set_index("month")["date_prev"]
needed = sorted({y for m in months for y in (days[m].year - 1, days[m].year)} & set(range(int(C.DAILY_FILE_START[:4]), int(C.PANEL_LAST_MONTH[:4]) + 1)))
rows_needed = ulong[ulong["month"].isin(months)]
daily, _ = L.daily_matrix(root, rows_needed["permno"].unique(), needed)
factors, risk_free, _ = D.french_daily()
raw_links, _ = D.read_table(root, "crsp_ccm_lnkhist")
funda, _ = D.read_table(root, "comp_funda")
inputs = {"monthly": monthly, "delistings": delistings, "daily": daily, "market_excess": factors["Mkt-RF"],
          "risk_free": risk_free, "links": raw_links, "funda": funda}
built = panel[["month", "permno", *names]]
lookahead = []
for m in months:
    rows = rows_needed[rows_needed["month"] == m]
    result = P.compare_month(built[built["month"] == m], P.build_cut_month(rows, inputs))
    lookahead.append({"month": str(m), "members": int(len(rows)), **result})
lookahead = pd.DataFrame(lookahead)
g1c_met = bool(lookahead["equal"].all())
print(lookahead.to_string(index=False))
print(f"{time.time() - t0:.0f} s | Gate G1 (c):", "met" if g1c_met else "not met",
      f"| largest difference {lookahead['largest_difference'].max()} | cells present in one build only {int(lookahead['presence_differs'].sum())}")

## 6. Gate G1

The three parts together. If any is not met, nothing after this notebook runs.

In [ ]:
g1 = {"g1a_met": bool(g1a["g1a_met"]), "g1b_met": bool(g1b["g1b_met"]), "g1c_met": g1c_met}
g1["g1_met"] = all(g1.values())
print("Gate G1:", "met" if g1["g1_met"] else "not met", json.dumps(g1))

## 7. What is kept and what is committed

The panel is kept in `local/` twice, before and after the transform, for milestone 2. Five files go into `results/20260928/`, each an aggregate with no identifier of a security: the missing shares by characteristic and year, gate G1 (b) with each characteristic's lowest month, the look-ahead test's 24 rows, gate G1 and a summary.

In [ ]:
D.write_local("panel.parquet", panel.assign(month=panel["month"].astype(str)))
D.write_local("panel_transformed.parquet", transformed.assign(month=transformed["month"].astype(str)))
written = [D.write_results("panel_missing_by_year.csv", frame=missing),
           D.write_results("g1b_coverage.json", record=g1b),
           D.write_results("g1c_lookahead.csv", frame=lookahead),
           D.write_results("g1.json", record=g1),
           D.write_results("panel_summary.json", record={
               "written_by": "notebooks/05_panel.ipynb", "manifest_sha256": C.MANIFEST_SHA256, **counts,
               "filled_cells": filled, "transform_checks": checks}),
           D.write_french_provenance(french_monthly_records)]
for path in written:
    print("written:", path.relative_to(ROOT))
results = pd.DataFrame([
    ("1. member-months, without a target, in the test window", f"{C.PANEL_MEMBER_MONTHS:,}, {C.PANEL_MEMBER_MONTHS_NO_RETURN}, {C.TEST_MEMBER_MONTHS:,}",
     f"{counts['member_months']:,}, {counts['without_target']}, {counts['test_member_months']:,}", "no verdict"),
    ("2. gate G1 (b), every characteristic at its level in every month", "met", "met" if g1b["g1b_met"] else "not met", "met" if g1b["g1b_met"] else "not met"),
    ("3. gate G1 (c), 24 months equal to 1e-12", "met, largest difference 0", f"largest difference {lookahead['largest_difference'].max()}", "met" if g1c_met else "not met"),
    ("4. the transform's three checks", "all true", str(all(checks.values())), "no verdict"),
    ("gate G1, all three parts", "met", "met" if g1["g1_met"] else "not met", "met" if g1["g1_met"] else "not met"),
], columns=["expectation", "expected", "result", "verdict"])
print(results.to_string(index=False))
if not g1["g1_met"]:
    raise SystemExit("Gate G1 is not met, so nothing after this notebook runs.")

## What was learned, and what could be wrong

- The table above sets each expectation beside its result and gives the gate's verdict.
- The look-ahead test shows that the code uses nothing dated after the last trading day of month t-1 in the 24 months it builds a second time. It cannot show that the data themselves were known on that day: CRSP and Compustat correct their history, and the extraction holds the corrected values. That limit is the data's, and no test on one extraction can lift it.
- The fill puts a member with a missing value at the month's median. A model can then learn nothing from that characteristic for that member, which is the purpose, but a characteristic missing for a quarter of the members in some month would weaken what any model learns from it in that month. The missing shares by year are committed for that reason.
- The rank transform keeps the order of the members within a month and drops the size of the differences. A model fitted on it forecasts from where a stock stands among the month's members, which is the treatment of Gu, Kelly and Xiu (2020) that the study follows.